# 04 — Query recipes

Twelve queries a researcher actually wants, each with its caveats inline. Copy, adapt, repoint.

**Three rules that apply to every recipe below:**
1. Count papers with `canonical_pmid.nunique()` — never `pmid` (double-counts 8 merged records) and
   never row counts (one paper contributes many findings).
2. `canonical_entity` covers only **29.8%** of rows (10,858 of 36,459; 2,185 of 3,121 papers) — the
   original `entity` string covers 53.0%. Any entity-keyed query drops **~70%** of the table, not
   half. 8,449 of the dropped rows *do* have an entity string that was never canonicalized: find
   them by keyword on `entity_norm_original`, not by entity rollup.
3. Filter `map_confidence == "high"` for load-bearing entity work (51 low + 240 medium mappings exist).

> ## Scope — read this first
>
> **This is Tier 1: POTS / dysautonomia literature ONLY** — chunks 0–75 of a 286-chunk corpus,
> **~3,121 of ~14,263 corpus papers**.
>
> ME/CFS, fibromyalgia, EDS/hypermobility, MCAS, IBS, migraine and autism appear here **only as
> incidental mentions inside POTS/dysautonomia papers**. Those conditions' own literatures are in
> Tiers 2–4 and are **not yet extracted**. This is **not** a cross-condition resource.
>
> **Not medical advice.** Machine-extracted research information, not clinical guidance. No
> diagnostic or treatment recommendations. Personal health questions belong with a qualified
> clinician.
>
> **Attribution.** Every finding links to its source paper with authors and journal. These are
> automated summaries of other researchers' work — the credit is theirs, and the original paper is
> the authority. `provenance_quote_trunc` is short verbatim publisher-copyright text included for
> verification only; always keep it next to its citation and never bulk-export it.

In [ ]:
# ── Data location ──────────────────────────────────────────────────────────────
# Repoint this constant at your own fork if you want. Raw GitHub is used in Colab;
# a local ./data or ../data directory (i.e. running inside a clone) wins if present.
RAW_BASE = "https://raw.githubusercontent.com/USER/REPO/main/data"

import os, io, json, urllib.request
import pandas as pd, numpy as np
import matplotlib
import matplotlib.pyplot as plt

_LOCAL = next((p for p in ("data", "../data", "../../data") if os.path.isdir(p)), None)
_CACHE = ".cache_tier1"; os.makedirs(_CACHE, exist_ok=True)

def data_path(name):
    """Return a readable path for a data file: local clone if present, else download once."""
    if _LOCAL and os.path.exists(os.path.join(_LOCAL, name)):
        return os.path.join(_LOCAL, name)
    dst = os.path.join(_CACHE, name)
    if not os.path.exists(dst):
        url = f"{RAW_BASE}/{name}"
        print(f"downloading {name} ...")
        urllib.request.urlretrieve(url, dst)
    return dst

def load(name):
    p = data_path(name)
    if p.endswith(".parquet"): return pd.read_parquet(p)
    if p.endswith(".json"):    return json.load(open(p))
    return pd.read_csv(p, low_memory=False)

print("data source:", _LOCAL if _LOCAL else RAW_BASE)

In [ ]:
F  = load("tier1_findings.parquet")
A  = load("association_table.parquet")
U  = load("understudied_signals.parquet")
C  = load("contradictions.parquet")
NL = load("null_results.parquet")
V  = load("entity_vocabulary.parquet")

def papers(df): return df.canonical_pmid.nunique()
def papers_direct(df): return df.loc[~df.split_derived, "canonical_pmid"].nunique()
CITE = ["citation_short","journal","year","pubmed_url","doi_url"]
print(f"{len(F):,} findings · {papers(F):,} papers")

## 1. All RCT- and meta-analysis-tier treatment failures

The most decision-relevant subset in the database. **Caveat:** "failed in this trial" is not
"ineffective" — check `comparator`, `n_participants` and `population_age_group` before concluding
anything, and read the source.

In [ ]:
q1 = F[(F.finding_type == "treatment/intervention")
       & F.evidence_tier.isin(["RCT","meta-analysis"])
       & F.null_or_failed_result.isin(["null-no-effect","failed-to-replicate","tried-and-abandoned"])]
print(f"{len(q1)} findings across {papers(q1)} papers")
print("\nby treatment entity:")
print(q1.groupby("canonical_entity").canonical_pmid.nunique().sort_values(ascending=False).head(10).to_string())
q1[["canonical_entity","treatment_class","direction","evidence_tier","n_participants","comparator"] + CITE].head(12)

## 2. Everything about one entity, with its papers

**Caveat:** match on `canonical_entity`, not the raw `entity` string — the raw strings have 10,184
variants. Use the vocabulary to find the canonical name first.

In [ ]:
TARGET = "norepinephrine"   # <- change me

print("vocabulary entries mapping to this canonical name:")
print(V[V.canonical_name == TARGET][["entity_norm","map_confidence","ontology_id","rows","papers"]].head(10).to_string(index=False))

q2 = F[F.canonical_entity == TARGET]
print(f"\n{len(q2)} findings · {papers(q2)} papers · {papers_direct(q2)} papers_direct (excl. split_derived)")
print(f"map_confidence: {dict(q2.map_confidence.value_counts())}")
print("\ndirection profile:"); print(q2.direction.value_counts().to_string())
print("\nsource papers (top 10 by findings):")
print(q2.groupby(["canonical_pmid","citation_short"]).size().sort_values(ascending=False).head(10).to_string())

## 3. Markers with ≥5 papers and ≥0.8 direction agreement

The "consistent signal" query. **Caveat:** direction agreement must be computed over *directional*
rows only — `not applicable` and `no significant difference` are not agreement.

In [ ]:
DIRECTIONAL = ~F.direction.isin(["not applicable","no significant difference","mixed"])
base = F[DIRECTIONAL & F.canonical_entity.notna() & (F.map_confidence == "high")]

b = base.copy()
b["_direct_pmid"] = b.canonical_pmid.where(~b.split_derived)
g = b.groupby("canonical_entity").agg(papers=("canonical_pmid","nunique"),
                                      papers_direct=("_direct_pmid","nunique"),
                                      findings=("finding_text","size"))
mode = (b.groupby(["canonical_entity","direction"]).size().rename("n").reset_index()
          .sort_values(["canonical_entity","n"], ascending=[True,False])
          .drop_duplicates("canonical_entity").set_index("canonical_entity"))
g["top_direction"] = mode.direction
g["direction_agreement"] = mode.n / g.findings
q3 = g[(g.papers >= 5) & (g.direction_agreement >= 0.8)].sort_values(["papers","direction_agreement"], ascending=False)
print(f"{len(q3)} entities with >=5 papers and >=0.8 direction agreement (high-confidence mappings only)")
q3.head(20)

## 4. Contradictions with a subtype (or other) moderator

The tractable disagreements — where a visible variable may explain the conflict. **Caveat:** check
`broad_condition_flag` too; 376 of 532 conflicts sit under umbrella labels and may be pooling
distinct diseases rather than failing to replicate.

In [ ]:
q4 = C[C.has_visible_moderator].sort_values("conflict_strength", ascending=False)
print(f"{len(q4)} of {len(C)} conflicts have a visible moderator")
print(f"of those, {int(q4.broad_condition_flag.sum())} ALSO carry broad_condition_flag (read with extra care)")
sub = q4[q4.moderator_candidates.fillna("").str.contains("subtype", case=False)]
print(f"\n{len(sub)} have a SUBTYPE moderator specifically:")
sub[["canonical_entity","condition","conflict_type","side_A_direction","n_papers_A",
     "side_B_direction","n_papers_B","moderator_detail"]].head(12)

## 5. Under-studied replication targets, filtered for independence

**Caveat:** thin evidence includes single-lab artifacts and extraction errors by construction.
This is a triage queue, not a list of neglected truths.

In [ ]:
q5 = U[U.headline_eligible & (U.n_distinct_last_authors >= 2) & ~U.case_report_only]
print(f"{len(q5)} replication targets (headline-eligible, >=2 independent last authors, not case-report-only)")
print(f"out of {len(U)} candidates; {int(U.headline_eligible.sum())} headline-eligible")
q5.sort_values("replication_score", ascending=False).head(15)[
    ["canonical_entity","entity_kind","condition","direction","n_papers","n_distinct_last_authors",
     "direction_agreement","best_tier","replication_score"]]

## 6. Biomarkers reported elevated in POTS specifically

**Caveat:** `condition == "POTS"` is narrower than the corpus — `dysautonomia` (18,293 findings)
and `general` (5,854) also cover POTS papers. Decide deliberately whether you want the strict label
or the union.

In [ ]:
strict = F[(F.condition == "POTS") & (F.direction == "elevated")
           & (F.finding_type == "mechanism/biomarker") & F.canonical_entity.notna()]
wide   = F[F.condition.isin(["POTS","dysautonomia","general"]) & (F.direction == "elevated")
           & (F.finding_type == "mechanism/biomarker") & F.canonical_entity.notna()]
print(f"strict (condition == 'POTS'): {len(strict)} findings, {papers(strict)} papers")
print(f"wide   (POTS|dysautonomia|general): {len(wide)} findings, {papers(wide)} papers")
print("\ntop elevated biomarkers, strict label:")
print(strict.groupby("canonical_entity").canonical_pmid.nunique().sort_values(ascending=False).head(15).to_string())

## 7. Findings from a specific study design, with participant counts

**Caveat:** `n_participants` is 53.9% filled and includes single-case counts. Check `evidence_tier`
before averaging anything.

In [ ]:
q7 = F[(F.evidence_tier == "RCT") & F.n_participants.notna()]
print(f"{len(q7)} RCT findings with a participant count, {papers(q7)} papers")
print(f"n_participants: median {q7.n_participants.median():.0f}, "
      f"IQR {q7.n_participants.quantile(.25):.0f}-{q7.n_participants.quantile(.75):.0f}, max {q7.n_participants.max():.0f}")
print(f"\nfill rate of n_participants overall: {100*F.n_participants.notna().mean():.1f}%")
q7[["canonical_entity","finding_type","n_participants","comparator","direction"] + CITE[:3]].head(10)

## 8. Pediatric findings

**Caveat:** `population_age_group` is `not-stated` for 8,467 rows — those are unknown, not adult.

In [ ]:
q8 = F[F.population_age_group == "pediatric"]
print(f"{len(q8)} findings, {papers(q8)} papers")
print(f"not-stated rows (unknown age, NOT adult): {int((F.population_age_group=='not-stated').sum()):,}")
print("\nfinding_type profile vs whole corpus:")
print(pd.DataFrame({"pediatric_%": 100*q8.finding_type.value_counts(normalize=True),
                    "all_%": 100*F.finding_type.value_counts(normalize=True)}).round(1))

## 9. Autoantibody and immune signals

**Caveat:** a keyword search over `canonical_entity` and `finding_text` will catch mentions in
passing, including negative statements. Check `direction` and `null_or_failed_result`.

In [ ]:
PAT = r"autoantib|adrenergic receptor|muscarinic|acetylcholine receptor|angiotensin ii type"
q9 = F[F.canonical_entity.fillna("").str.contains(PAT, case=False, regex=True)
       | F.finding_text.fillna("").str.contains(PAT, case=False, regex=True)]
print(f"{len(q9)} findings, {papers(q9)} papers")
print("\ndirection profile - note the negatives:")
print(q9.direction.value_counts().to_string())
print("\nnull_or_failed_result:"); print(q9.null_or_failed_result.value_counts().to_string())
print("\ntop canonical entities:")
print(q9[q9.canonical_entity.notna()].groupby("canonical_entity").canonical_pmid.nunique().sort_values(ascending=False).head(10).to_string())

## 10. Long-COVID vs non-COVID dysautonomia

**Caveat:** `is_covid_associated` is a finding-level flag. This compares how the two literatures
are *reported*, not the two patient populations.

In [ ]:
cov, non = F[F.is_covid_associated], F[~F.is_covid_associated]
print(f"COVID-associated: {len(cov):,} findings, {papers(cov):,} papers")
print(f"not COVID-assoc. : {len(non):,} findings, {papers(non):,} papers")
comp = pd.DataFrame({"covid_%": 100*cov.finding_type.value_counts(normalize=True),
                     "non_covid_%": 100*non.finding_type.value_counts(normalize=True)}).round(1)
comp["difference"] = comp["covid_%"] - comp["non_covid_%"]
print(); print(comp)

## 11. Papers with the most findings — and a caution about them

**Caveat:** a high finding count usually means a *review*, not a strong primary study. Check
`evidence_tier` before treating density as importance.

In [ ]:
q11 = (F.groupby(["canonical_pmid","citation_short","journal","year"])
        .agg(findings=("finding_text","size"),
             top_tier=("evidence_tier", lambda s: s.value_counts().index[0]))
        .sort_values("findings", ascending=False).head(15).reset_index())
print(q11[["citation_short","journal","year","findings","top_tier"]].to_string(index=False))
print("\nfindings-per-paper: median",
      int(F.groupby('canonical_pmid').size().median()), "| mean",
      round(F.groupby('canonical_pmid').size().mean(), 1))

## 12. Build a verifiable evidence table for one claim

The pattern to use whenever a finding is going into your own writing: pull the rows, keep the
citation and the short quote adjacent, then **read the sources**. The quote is a locator, not a
substitute for the paper.

In [ ]:
ENTITY, DIRECTION = "heart rate variability", "depleted/reduced"
ev = F[(F.canonical_entity == ENTITY) & (F.direction == DIRECTION)][
    ["canonical_pmid","citation_short","journal","year","evidence_tier","certainty","n_participants",
     "condition","finding_text","provenance_section","provenance_quote_trunc","pubmed_url"]]
print(f"{len(ev)} findings across {ev.canonical_pmid.nunique()} papers for '{ENTITY}' / '{DIRECTION}'")
print("tier profile:", dict(ev.evidence_tier.value_counts()))
ev.to_csv("evidence_table.csv", index=False)
print("\nwrote evidence_table.csv - quotes are <=200 chars and kept beside their citation and link.")
for _, r in ev.head(3).iterrows():
    print(f"\n{r.citation_short} | {r.evidence_tier} | {r.provenance_section}")
    print(f"  finding: {r.finding_text[:160]}")
    print(f"  quote  : {str(r.provenance_quote_trunc)[:160]}")
    print(f"  source : {r.pubmed_url}")

## Where these queries can mislead you

- **Absence is not absence.** Tier 1 is ~3,121 of ~14,263 corpus papers. "No results" means "not in
  Tier 1."
- **Entity queries cover ~30% of the table, not half.** `canonical_entity` is present on 10,858 of
  36,459 rows. Of the rest, 17,152 name no entity at all (by design) and **8,449 carry an entity
  string that was never canonicalized** — those are real findings that an entity rollup silently
  excludes. Search them by keyword on `entity` / `entity_norm_original` before concluding the
  corpus says nothing about a marker.
- **State your pmid basis.** Counts here use `canonical_pmid`. The raw-`pmid` basis differs for 3
  of 863 entities (elp1 58 vs 53 papers, norepinephrine 195 vs 194, ivabradine 63 vs 62); the
  `papers_direct` basis differs again. Don't mix bases in one table.
- **Condition labels are not populations.** Non-POTS labels count incidental mentions inside
  POTS/dysautonomia papers.
- **Review articles inflate counts.** `review/expert-opinion` is 6,507 findings; reviews describe
  other studies' work, and that is a known extraction error mode.
- **Every row is machine-extracted.** Use `provenance_quote_trunc` + `pubmed_url` to check any
  finding you intend to rely on, and cite the original paper — not this database.